<a href="https://colab.research.google.com/github/gaurav-0022/Assignment-1/blob/main/Data_Cleaning_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import files
uploaded = files.upload()

Saving healthcare_data_cleaning_dataset.csv to healthcare_data_cleaning_dataset.csv


In [4]:
import pandas as pd

df = pd.read_csv('healthcare_data_cleaning_dataset.csv')
df.head()

,Patient_ID,Age,Gender,City,Diagnosis,Hospital_Visits,Treatment_Cost,Insurance_Coverage,Admission_Date
0,17270,35.0,Male,Bangalore,Hypertension,13,41010.0,1,2023-11-30
1,10860,21.0,Female,Hyderabad,Flu,11,12194.0,1,2023-02-23
2,15390,77.0,Female,Bangalore,Asthma,2,45086.0,0,2023-03-14
3,15191,79.0,Female,Mumbai,Asthma,13,40842.0,0,2023-08-01
4,15734,60.0,Female,Delhi,Asthma,1,9873.0,1,2023-06-20


##Q1. Missing Data Identification

Scenario:
 The hospital suspects incomplete patient records.

Task:

Identify missing values in each column

Calculate percentage of missing data

In [5]:
# Count missing values
missing_count = df.isnull().sum()

# Percentage of missing values
missing_percent = (df.isnull().sum() / len(df)) * 100

# Combine results
missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_percent
})

missing_df

,Missing Count,Missing %
Patient_ID,0,0.000000
Age,600,11.764706
Gender,0,0.000000
City,0,0.000000
Diagnosis,0,0.000000
Hospital_Visits,0,0.000000
Treatment_Cost,593,11.627451
Insurance_Coverage,0,0.000000
Admission_Date,0,0.000000


##Q2. Handling Missing Age

Scenario:
 Age is critical for medical analysis, but some values are missing.

Task:

Replace missing Age values with an appropriate method

Justify your choice (mean/median)

In [6]:
# Check distribution
df['Age'].describe()

# Replace with median
df['Age'].fillna(df['Age'].median(), inplace=True)

/tmp/ipykernel_11150/3898466360.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)


##Q3. Handling Missing Treatment Cost

Scenario:
 Treatment cost is highly skewed due to expensive treatments.

Task:

Handle missing Treatment_Cost values

Choose the correct imputation method and explain why

In [7]:
# Check skewness
df['Treatment_Cost'].skew()

# Fill with median
df['Treatment_Cost'].fillna(df['Treatment_Cost'].median(), inplace=True)

/tmp/ipykernel_11150/2824044227.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Treatment_Cost'].fillna(df['Treatment_Cost'].median(), inplace=True)


##Q4. Duplicate Patient Records

Scenario:
 Some patient records were entered multiple times.

Task:

Identify duplicate rows

Remove duplicates

Compare dataset size before and after

In [8]:
# Before
before = df.shape[0]

# Find duplicates
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)

# Remove duplicates
df = df.drop_duplicates()

# After
after = df.shape[0]

print("Rows before:", before)
print("Rows after:", after)

Duplicate rows: 99
Rows before: 5100
Rows after: 5001


##Q5. Invalid Age Values (Data Quality Check)

Scenario:
 Some patients have unrealistic age values (e.g., >100 or <0).

Task:

Detect such records

Decide whether to remove or correct them



In [9]:
# Detect invalid ages
invalid_age = df[(df['Age'] < 0) | (df['Age'] > 100)]
print("Invalid records:", len(invalid_age))

# Remove them
df = df[(df['Age'] >= 0) & (df['Age'] <= 100)]

Invalid records: 0


##Q6. Outlier Detection (Treatment Cost)

Scenario:
 Extreme treatment costs are affecting analysis.

Task:

Detect outliers using IQR method

Display number of outliers

In [10]:
Q1 = df['Treatment_Cost'].quantile(0.25)
Q3 = df['Treatment_Cost'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['Treatment_Cost'] < lower) | (df['Treatment_Cost'] > upper)]

print("Number of outliers:", len(outliers))

Number of outliers: 50


##Q7. Outlier Treatment

Scenario:
The business team wants to retain all records.

Task:

Apply capping (Winsorization) on Treatment_Cost

Use 5th and 95th percentile

In [17]:
import numpy as np

# Check column name first
print(df.columns)

# Calculate percentiles
lower_cap = df['Treatment_Cost'].quantile(0.05)
upper_cap = df['Treatment_Cost'].quantile(0.95)

# Apply capping (Winsorization)
df['Treatment_Cost'] = df['Treatment_Cost'].clip(lower=lower_cap, upper=upper_cap)

# Check result
df['Treatment_Cost'].describe()

Index(['Patient_ID', 'Age', 'Gender', 'City', 'Diagnosis', 'Hospital_Visits',
       'Treatment_Cost', 'Insurance_Coverage', 'Admission_Date',
       'Log_Treatment_Cost'],
      dtype='object')


,Treatment_Cost
count,5001.000000
mean,25218.878624
std,13580.445490
min,3238.000000
25%,13766.000000
50%,24797.000000
75%,36542.000000
max,47948.000000


##Q8. Transformation

Scenario:
 Treatment cost is highly skewed.

Task:

Apply log transformation

Create a new column

Compare before vs after distribution

In [15]:
# Create new column
df['Log_Treatment_Cost'] = np.log1p(df['Treatment_Cost'])

# Compare distributions
print("Before skew:", df['Treatment_Cost'].skew())
print("After skew:", df['Log_Treatment_Cost'].skew())

Before skew: 0.05543898196772683
After skew: -1.0348951999711247


##Q9. Time-Based Missing Handling

Scenario:
 Admission dates should follow a logical sequence.

Task:

Sort data by Admission_Date

Apply forward fill or backward fill where appropriate

Justify your choice


In [16]:
# Convert to datetime
df['Admission_Date'] = pd.to_datetime(df['Admission_Date'])

# Sort by date
df = df.sort_values(by='Admission_Date')

# Forward fill
df.fillna(method='ffill', inplace=True)

/tmp/ipykernel_11150/1239806063.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
